# DropoutIA — Modélisation : comparaison de plusieurs algorithmes

Ce notebook compare **4 modèles de Machine Learning**, du plus simple au
plus complexe, sur le jeu de données du **décrochage** (`Resultat`, binaire).

Vous pourrez dupliquer ce notebook (Semaine 3) pour faire la même
comparaison sur le jeu de données du **niveau de risque**
(`id_niveauRisqueInit`, 4 catégories) — le pipeline est identique, seul le
fichier chargé et la colonne cible changent.

**Rappel du principe** : on ne compare pas les modèles au hasard, on suit un
palier de complexité croissante pour comprendre à chaque étape *pourquoi*
un modèle fait mieux (ou pas) que le précédent.

## 1. Chargement des données nettoyées

In [ ]:
import pandas as pd
import sys
sys.path.append("../src")
import modelisation

df = pd.read_csv("../data/processed/donnees_clean.csv")
print(df.shape)
df.head()

## 2. Préparation train/test

`preparer_donnees` sépare X (caractéristiques) et y (la cible `Resultat`),
puis crée un ensemble d'entraînement (80%) et un ensemble de test (20%).
Voir les commentaires dans `src/modelisation.py` pour le détail de
`stratify` et pourquoi c'est important avec des classes déséquilibrées.

In [ ]:
X_train, X_test, y_train, y_test = modelisation.preparer_donnees(df, colonne_cible="Resultat")
print("Train :", X_train.shape, "| Test :", X_test.shape)
print("\nProportion de decrocheurs dans train :", round(y_train.mean()*100, 2), "%")
print("Proportion de decrocheurs dans test  :", round(y_test.mean()*100, 2), "%")

## 3. Comparaison des 4 modèles

- **Régression logistique** : le modèle le plus simple, sert de référence (baseline).
- **Arbre de décision** : un seul arbre, facile à interpréter mais souvent moins précis.
- **Random Forest** : combine 200 arbres différents, réduit le surapprentissage.
- **Gradient Boosting** : les arbres se corrigent successivement les uns les autres,
  souvent la meilleure performance sur des données tabulaires comme les vôtres.

On regarde 4 métriques, pas juste l'accuracy :
- **Accuracy** : % de bonnes prédictions global (trompeur si déséquilibre).
- **Precision** : parmi les élèves prédits "décrocheurs", combien le sont vraiment.
- **Recall** : parmi les vrais décrocheurs, combien le modèle en détecte.
- **F1-score** : équilibre entre precision et recall — la métrique à privilégier ici.

In [ ]:
tableau_resultats, modeles_entraines = modelisation.comparer_modeles(X_train, X_test, y_train, y_test)
tableau_resultats

## 4. Analyse détaillée du meilleur modèle

Repérez le nom du modèle en tête du tableau ci-dessus, et mettez-le dans la
variable `meilleur_nom` ci-dessous pour voir son rapport détaillé.

In [ ]:
meilleur_nom = tableau_resultats.iloc[0]['modele']  # récupère automatiquement le 1er du classement
print("Meilleur modele :", meilleur_nom)

meilleur_modele = modeles_entraines[meilleur_nom]
modelisation.rapport_detaille(meilleur_modele, X_test, y_test)

## 5. Importance des caractéristiques (interprétabilité)

Quelles colonnes pèsent le plus dans la décision du modèle ? Très utile
pour votre rapport ("le modèle se base principalement sur X et Y pour
détecter le risque"). Ne fonctionne que pour Random Forest / Gradient
Boosting (pas pour la régression logistique, qui a une autre notion
d'importance -- ses coefficients).

In [ ]:
import matplotlib.pyplot as plt

if hasattr(meilleur_modele, 'feature_importances_'):
    importances = pd.Series(meilleur_modele.feature_importances_, index=X_train.columns)
    importances.sort_values(ascending=True).plot(kind='barh', figsize=(8,6))
    plt.title(f"Importance des caracteristiques - {meilleur_nom}")
    plt.tight_layout()
    plt.show()
else:
    print("Ce modele n'a pas d'attribut feature_importances_ (normal pour la regression logistique).")

## 6. Sauvegarde du meilleur modèle

On sauvegarde uniquement le meilleur modèle (pas les 4) — c'est celui-ci
qui sera utilisé plus tard par l'API.

In [ ]:
import joblib
joblib.dump(meilleur_modele, "../models/modele_decrochage.pkl")
print("Modele sauvegarde :", meilleur_nom)